In [3]:
from pathlib import Path
import pymupdf
from IPython.display import Image, display
from pathlib import Path
import torch
import transformers
from PIL import Image
from transformers import pipeline


c:\Users\nico_\Desktop\rag_pdf\venvrag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
DATA_DIR = Path("../data/raw")
list(DATA_DIR.glob("*.pdf"))

[WindowsPath('../data/raw/microsoft_report.pdf'),
 WindowsPath('../data/raw/test.pdf'),
 WindowsPath('../data/raw/worldbank_report.pdf')]

In [5]:
pdf_path = next(DATA_DIR.glob("*test*")) # next() pour récupérer la valeur de l'objet créer par glob()

print(pdf_path)

..\data\raw\test.pdf


In [6]:
doc = pymupdf.open(pdf_path)

print(f"Nombre de pages : {len(doc)}")
print(f"Métadonnées : {doc.metadata}")

Nombre de pages : 2
Métadonnées : {'format': 'PDF 1.4', 'title': 'Document sans titre', 'author': '', 'subject': '', 'keywords': '', 'creator': '', 'producer': 'Skia/PDF m155 Google Docs Renderer', 'creationDate': '', 'modDate': '', 'trapped': '', 'encryption': None}


In [7]:
for numero, page in enumerate(doc):
    print(f"\n--- Page {numero + 1} ---")

    ### Texte
    texte = page.get_text("text")
    print("Nombre de caractères :", len(texte))

    ### Tableau
    resultats = page.find_tables()
    print("Nombre de tableaux détectés :", len(resultats.tables))

    for i, tableau in enumerate(resultats.tables):
        print(
            f"  Tableau {i} : "
            f"{tableau.row_count} lignes, "
            f"{tableau.col_count} colonnes")

    ### Image
    image_list = page.get_images()
    print("Nombre d'image détectées :", len(image_list))

    for image_index, img in enumerate(image_list, start=1):
        xref = img[0]
        largeur = img[2]
        hauteur = img[3]

        positions = page.get_image_rects(xref)

        print(f"\nImage {image_index}")
        print("  xref :", xref)
        print("  dimensions :", largeur, "x", hauteur)
        print("  positions :", positions)


--- Page 1 ---
Nombre de caractères : 1195
Consider using the pymupdf_layout package for a greatly improved page layout analysis.
Nombre de tableaux détectés : 1
  Tableau 0 : 20 lignes, 3 colonnes
Nombre d'image détectées : 0

--- Page 2 ---
Nombre de caractères : 14
Nombre de tableaux détectés : 0
Nombre d'image détectées : 2

Image 1
  xref : 9
  dimensions : 200 x 200
  positions : [Rect(73.5, 66.9749755859375, 215.25, 208.7249755859375)]

Image 2
  xref : 11
  dimensions : 855 x 535
  positions : [Rect(73.5, 278.3123779296875, 525.0, 560.3123779296875)]


Doc PDF -> Détection des éléments :

-> Texte -> Chunking textuel

-> Tableau -> Chunking tabulaire

-> Images, graphiques -> OCR ou modèle de vision

-> Index vectoriel

-> Réponse aux sources

-> Surlignage par bbox

# Tableau

In [8]:
tableaux_document = [] # Liste de tableaux par document

for numero, page in enumerate(doc):
    #print(f"\n--- Page {numero + 1} ---")

    resultats = page.find_tables()
    #print("Nombre de tableaux détectés :", len(resultats.tables))

    for i, tableau in enumerate(resultats.tables):

            # Extraction des coordonnées du tableau

        bbox_tableau = list(tableau.bbox)

        geometrie_entete = tableau.rows[0]
        bbox_entete = list(geometrie_entete.bbox)

        donnees = tableau.extract() # Extraction du contenu du tableau. Les donnees deviennent une liste contenant toutes les lignes 

        if donnees:
            titres_colonnes = donnees[0]

            lignes_donnees = donnees[1:]

            if lignes_donnees:

                lignes_structurees = []

                for numero_ligne, ligne in enumerate(lignes_donnees,start=1): # start=1 pour correspondre aux index de tableau.rows car tableau.rows[0] contient l'en-tête

                    # Association des titres avec les valeurs
                    donnees_ligne = dict(zip(titres_colonnes, ligne)) # # zip() associe les éléments qui ont la même position
                    
                    # Géométrie ligne actuelle
                    geometrie_ligne = tableau.rows[numero_ligne]

                    # Bbox de la ligne complète
                    bbox_ligne = list(geometrie_ligne.bbox)
                    
                    # Liste des cellules de la ligne actuelle
                    cellules_structurees = []

                    for numero_colonne, titre in enumerate(titres_colonnes):

                        valeur = ligne[numero_colonne]

                        bbox_cellule = geometrie_ligne.cells[numero_colonne]

                        cellule_structuree = {
                            "colonne": titre,
                            "valeur": valeur,
                            "bbox": (
                                list(bbox_cellule) 
                                if bbox_cellule is not None
                                else None
                            )
                        }

                        cellules_structurees.append(cellule_structuree)

                    # Création de la ligne après avoir traité ses cellules
                    ligne_structuree = {
                        "data": donnees_ligne,
                        "bbox": bbox_ligne,
                        "cellules": cellules_structurees
                    }

                    # Ajout de la ligne terminée au tableau
                    lignes_structurees.append(ligne_structuree)
                
                # Construction d'un tableau complet
                tableau_structure = {
                    "type": "tableau",
                    "page": numero +1,
                    "bbox": bbox_tableau,
                    "colonnes": titres_colonnes,
                    "bbox_entete": bbox_entete,
                    "lignes": lignes_structurees

                }

                # Ajout du tableau à la liste du docume,nt
                tableaux_document.append(tableau_structure)


print(
    "Nombre total de tableaux :",
    len(tableaux_document)
)

premiere_ligne_structuree = (
    tableaux_document[0]["lignes"][0]
)

print(
    "Données de la première ligne :",
    premiere_ligne_structuree["data"]
)

print(
    "Bbox de la première ligne :",
    premiere_ligne_structuree["bbox"]
)

print(
    "Nombre de cellules :",
    len(premiere_ligne_structuree["cellules"])
)
    

Nombre total de tableaux : 1
Données de la première ligne : {'a': '1', 'b': '1', 'c': '1'}
Bbox de la première ligne : [72.5, 255.5, 297.5, 271.5]
Nombre de cellules : 3


In [9]:
print(tableaux_document[0]['type'])
print(tableaux_document[0]['page'])
print(tableaux_document[0]['bbox'])
print(tableaux_document[0]['colonnes'])
print(tableaux_document[0]['bbox_entete'])
print(tableaux_document[0]['lignes'])

tableau
1
[72.5, 238.5, 297.5, 568.5]
['a', 'b', 'c']
[72.5, 238.5, 297.5, 255.5]
[{'data': {'a': '1', 'b': '1', 'c': '1'}, 'bbox': [72.5, 255.5, 297.5, 271.5], 'cellules': [{'colonne': 'a', 'valeur': '1', 'bbox': [72.5, 255.5, 147.5, 271.5]}, {'colonne': 'b', 'valeur': '1', 'bbox': [147.5, 255.5, 222.5, 271.5]}, {'colonne': 'c', 'valeur': '1', 'bbox': [222.5, 255.5, 297.5, 271.5]}]}, {'data': {'a': '2', 'b': '2', 'c': '2'}, 'bbox': [72.5, 271.5, 297.5, 288.5], 'cellules': [{'colonne': 'a', 'valeur': '2', 'bbox': [72.5, 271.5, 147.5, 288.5]}, {'colonne': 'b', 'valeur': '2', 'bbox': [147.5, 271.5, 222.5, 288.5]}, {'colonne': 'c', 'valeur': '2', 'bbox': [222.5, 271.5, 297.5, 288.5]}]}, {'data': {'a': '3', 'b': '3', 'c': '3'}, 'bbox': [72.5, 288.5, 297.5, 304.5], 'cellules': [{'colonne': 'a', 'valeur': '3', 'bbox': [72.5, 288.5, 147.5, 304.5]}, {'colonne': 'b', 'valeur': '3', 'bbox': [147.5, 288.5, 222.5, 304.5]}, {'colonne': 'c', 'valeur': '3', 'bbox': [222.5, 288.5, 297.5, 304.5]}]}, {'

In [10]:
# Première ligne de valeur du tableau
premiere_ligne = tableaux_document[0]["lignes"][0]
print("Données :", premiere_ligne["data"])
print("Bbox :", premiere_ligne["bbox"])
print("Cellules :", premiere_ligne["cellules"])

Données : {'a': '1', 'b': '1', 'c': '1'}
Bbox : [72.5, 255.5, 297.5, 271.5]
Cellules : [{'colonne': 'a', 'valeur': '1', 'bbox': [72.5, 255.5, 147.5, 271.5]}, {'colonne': 'b', 'valeur': '1', 'bbox': [147.5, 255.5, 222.5, 271.5]}, {'colonne': 'c', 'valeur': '1', 'bbox': [222.5, 255.5, 297.5, 271.5]}]


# Texte

In [11]:
for numero, page in enumerate(doc):
    blocs = page.get_text("blocks", sort=True)

    print(blocs)### ###rrvergzerrvdrgERDFGBFGNFC CV SGS###737

[(272.09112548828125, 50.260597229003906, 328.74444580078125, 72.6043472290039, 'TEST \n', 0, 0), (72.0, 87.3157730102539, 280.3662414550781, 96.2532730102539, 'Qu’est-ce que la génération à enrichissement contextuel ? \n', 1, 0), (72.0, 110.27476501464844, 505.14263916015625, 119.21226501464844, 'La génération à enrichissement contextuel (RAG) est le processus consistant à optimiser le résultat d’un grand modèle de \n', 2, 0), (72.0, 124.99349975585938, 509.201171875, 133.93099975585938, 'langage. Elle fait donc appel à une base de connaissances fiable externe aux sources de données utilisées pour l’entraîner \n', 3, 0), (72.0, 139.71224975585938, 525.177734375, 148.64974975585938, 'avant de générer une réponse. Les grands modèles de langage (LLM) sont entraînés avec d’importants volumes de données et \n', 4, 0), (72.0, 154.43099975585938, 522.0401611328125, 163.36849975585938, 'utilisent des milliards de paramètres pour générer des résultats originaux pour des tâches telles que répon

In [12]:
blocs_texte_bruts = []

for numero, page in enumerate(doc):

    # Récupération du texte sous forme de blocs avec coordonnées, dans l'ordre de lecture
    blocs = page.get_text("blocks", sort=True)
    
    for bloc in blocs:
        x0, y0, x1, y1, contenu, numero_bloc, type_bloc = bloc

        if type_bloc == 0 and contenu.strip(): # == 0 pour conserver uniquement les blocs textuels / .strip() élimine les blocs vides
            bloc_texte = {
                "page": numero +1,
                "content": contenu.strip(),
                "bbox": [x0, y0, x1, y1]
            }

            blocs_texte_bruts.append(bloc_texte)

In [13]:
print(blocs_texte_bruts)

[{'page': 1, 'content': 'TEST', 'bbox': [272.09112548828125, 50.260597229003906, 328.74444580078125, 72.6043472290039]}, {'page': 1, 'content': 'Qu’est-ce que la génération à enrichissement contextuel ?', 'bbox': [72.0, 87.3157730102539, 280.3662414550781, 96.2532730102539]}, {'page': 1, 'content': 'La génération à enrichissement contextuel (RAG) est le processus consistant à optimiser le résultat d’un grand modèle de', 'bbox': [72.0, 110.27476501464844, 505.14263916015625, 119.21226501464844]}, {'page': 1, 'content': 'langage. Elle fait donc appel à une base de connaissances fiable externe aux sources de données utilisées pour l’entraîner', 'bbox': [72.0, 124.99349975585938, 509.201171875, 133.93099975585938]}, {'page': 1, 'content': 'avant de générer une réponse. Les grands modèles de langage (LLM) sont entraînés avec d’importants volumes de données et', 'bbox': [72.0, 139.71224975585938, 525.177734375, 148.64974975585938]}, {'page': 1, 'content': 'utilisent des milliards de paramètr

## Elimination des blocs textuels appartenant au tableau

In [14]:
blocs_texte_hors_tableaux = []

# Récupération bbox blocs
for bloc in blocs_texte_bruts:

    # Transformation bbox blocs texte en rectangle PyMuPDF
    rectangle_bloc = pymupdf.Rect(bloc["bbox"])

    # Variable appartenance au tableau
    bloc_dans_tableau = False

    # Comparaison des mêmes pages
    for tableau in tableaux_document:
        if tableau["page"] != bloc["page"]:
            continue
        
        # Transformation bbox tableau en rectangle PyMuPDF
        rectangle_tableau = pymupdf.Rect(
            tableau["bbox"]
        )

        # Vérification si intersection entre les deux rectangles
        if rectangle_tableau.intersects(rectangle_bloc):
            bloc_dans_tableau = True
            break
    
    # Conservation du bloc s'il n'est pas dans le tableau
    if not bloc_dans_tableau:
        blocs_texte_hors_tableaux.append(
            bloc
        )

In [15]:
print(
    "Blocs avant filtrage :",
    len(blocs_texte_bruts)
)

print(
    "Blocs après filtrage :",
    len(blocs_texte_hors_tableaux)
)

Blocs avant filtrage : 30
Blocs après filtrage : 10


In [16]:
for bloc in blocs_texte_hors_tableaux:
    print("\nPage :", bloc["page"])
    print("Bbox :", bloc["bbox"])
    print("Texte :", repr(bloc["content"]))


Page : 1
Bbox : [272.09112548828125, 50.260597229003906, 328.74444580078125, 72.6043472290039]
Texte : 'TEST'

Page : 1
Bbox : [72.0, 87.3157730102539, 280.3662414550781, 96.2532730102539]
Texte : 'Qu’est-ce que la génération à enrichissement contextuel ?'

Page : 1
Bbox : [72.0, 110.27476501464844, 505.14263916015625, 119.21226501464844]
Texte : 'La génération à enrichissement contextuel (RAG) est le processus consistant à optimiser le résultat d’un grand modèle de'

Page : 1
Bbox : [72.0, 124.99349975585938, 509.201171875, 133.93099975585938]
Texte : 'langage. Elle fait donc appel à une base de connaissances fiable externe aux sources de données utilisées pour l’entraîner'

Page : 1
Bbox : [72.0, 139.71224975585938, 525.177734375, 148.64974975585938]
Texte : 'avant de générer une réponse. Les grands modèles de langage (LLM) sont entraînés avec d’importants volumes de données et'

Page : 1
Bbox : [72.0, 154.43099975585938, 522.0401611328125, 163.36849975585938]
Texte : 'utilisent des

In [17]:
textes_document = []

for bloc in blocs_texte_hors_tableaux:
    texte_structure = {
        "type": "texte",
        "page": bloc["page"],
        "content": bloc["content"],
        "bbox": bloc["bbox"]
    }

    textes_document.append(
        texte_structure
    )

In [18]:
print(
    "Nombre de blocs de texte :",
    len(textes_document)
)

print(
    "Premier bloc :",
    textes_document[0]
)

Nombre de blocs de texte : 10
Premier bloc : {'type': 'texte', 'page': 1, 'content': 'TEST', 'bbox': [272.09112548828125, 50.260597229003906, 328.74444580078125, 72.6043472290039]}


# Images

In [19]:
for numero, page in enumerate(doc):
    images = page.get_images(full=True)

    for numero_image, image in enumerate(images):
        print(image)

(9, 0, 200, 200, 8, 'ICCBased', '', 'X9', 'DCTDecode', 0)
(11, 0, 855, 535, 8, 'ICCBased', '', 'X11', 'FlateDecode', 0)


In [20]:
for numero, page in enumerate(doc):
    images = page.get_images(full=True)

    for numero_image, image in enumerate(images):
        xref = image[0] # identifiant interne
        largeur = image[2]
        hauteur = image[3]

        positions = page.get_image_rects(xref)

        for position in positions:
            print(position)

Rect(73.5, 66.9749755859375, 215.25, 208.7249755859375)
Rect(73.5, 278.3123779296875, 525.0, 560.3123779296875)


In [21]:
# Liste contenant toutes les images du pdf
images_document = []

for numero, page in enumerate(doc):
    images = page.get_images(full=True)

    for numero_image, image in enumerate(images):
        xref = image[0] # identifiant interne
        largeur = image[2]
        hauteur = image[3]

        positions = page.get_image_rects(xref)

        for position in positions:
            image_structure = {
                "type": "image_inconnue", # image/schéma/graphique
                "page": numero+1,
                "image_index": numero_image,
                "xref": xref,
                "largeur": largeur,
                "hauteur": hauteur,
                "bbox": list(position)
            }

            images_document.append(image_structure)

In [22]:
print("Nombre total d'images :",len(images_document))

for image in images_document:
    print("\nType :", image["type"])
    print("Page :", image["page"])
    print("Xref :", image["xref"])
    print(
        "Dimensions :",
        image["largeur"],
        "x",
        image["hauteur"]
    )
    print("Bbox :", image["bbox"])

Nombre total d'images : 2

Type : image_inconnue
Page : 2
Xref : 9
Dimensions : 200 x 200
Bbox : [73.5, 66.9749755859375, 215.25, 208.7249755859375]

Type : image_inconnue
Page : 2
Xref : 11
Dimensions : 855 x 535
Bbox : [73.5, 278.3123779296875, 525.0, 560.3123779296875]


In [23]:
# Création  du dossier
dossier_images = Path("images_extraites")

dossier_images.mkdir(exist_ok=True)

In [24]:
for image in images_document:
    xref = image["xref"]

    # Récupération de l'image originale associée à l'identifiant
    image_extraite = doc.extract_image(xref)

    extension = image_extraite["ext"] # png ou jpeg
    contenu = image_extraite["image"] # données binaires de l'image

    mon_fichier = (
        f"page_{image['page']}"
        f"_xref_{xref}"
        f".{extension}"
    )

    chemin = dossier_images / mon_fichier

    # Enregistrement des données dans un fichier
    chemin.write_bytes(contenu)

    # Ajout du chemin du fichier dans la structure
    image["chemin"] = str(chemin)

In [25]:
for image in images_document:
    print("Xref :", image["xref"], "| Fichier :", image["chemin"])

Xref : 9 | Fichier : images_extraites\page_2_xref_9.jpeg
Xref : 11 | Fichier : images_extraites\page_2_xref_11.png


## Classification des images

In [26]:
#pip install "transformers[torch]" pillow

In [27]:
print("PyTorch :", torch.__version__)
print("Transformers :", transformers.__version__)
print("CUDA disponible :", torch.cuda.is_available())

PyTorch : 2.14.0+cpu
Transformers : 5.17.0
CUDA disponible : False


In [28]:
classifieur_image = pipeline(
    task="zero-shot-image-classification",
    model="openai/clip-vit-base-patch32",
    device=-1 # cpu
)

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 49884.44it/s]


In [29]:
categories = [
    "a logo",
    "a scanned document",
    "a chart or graph",
    "a diagram or schematic",
    "a photograph"
]

premiere_image = images_document[0]

predictions = classifieur_image(
    premiere_image["chemin"],
    candidate_labels=categories,
    hypothesis_template="This image is {}."
)

In [30]:
for prediction in predictions:
    print(prediction["label"],":",round(prediction["score"],3))

a logo : 0.924
a scanned document : 0.056
a diagram or schematic : 0.013
a chart or graph : 0.006
a photograph : 0.001


In [31]:
premiere_image = images_document[1]

predictions = classifieur_image(
    premiere_image["chemin"],
    candidate_labels=categories,
    hypothesis_template="Cette image est {}."
)

for prediction in predictions:
    print(prediction["label"],":",round(prediction["score"],3))

a chart or graph : 0.919
a diagram or schematic : 0.061
a logo : 0.008
a scanned document : 0.006
a photograph : 0.005


In [32]:
traduction_types = {
    "a logo": "logo",
    "a scanned document": "document_scanne",
    "a chart or graph": "graphique",
    "a diagram or schematic": "schema",
    "a photograph": "photographie"
}

for image in images_document:
    predictions = classifieur_image(
        image["chemin"],
        candidate_labels=list(
            traduction_types.keys()
        ),
        hypothesis_template="This image is {}."
    )

    meilleure_prediction = predictions[0] # Du meilleur score [0] au plus petit

    classe_anglaise = meilleure_prediction["label"]

    image["type"] = traduction_types[classe_anglaise] # Utilisation de la clé pour récupérer la valeur en français

    image["score_classification"] = meilleure_prediction["score"]

    image["predictions"] = predictions

In [33]:
for image in images_document:
    print("\nFichier :", image["chemin"])
    print("Type :", image["type"])
    print("Score :",round(image["score_classification"],3))


Fichier : images_extraites\page_2_xref_9.jpeg
Type : logo
Score : 0.924

Fichier : images_extraites\page_2_xref_11.png
Type : graphique
Score : 0.967


## OCR

In [34]:
%pip install easyocr

Note: you may need to restart the kernel to use updated packages.


In [35]:
import easyocr

print(easyocr.__version__)

1.7.2


In [36]:
# Initialisation du lecteur
lecteur_ocr = easyocr.Reader(["fr", "en"],gpu=False)

Using CPU. Note: This module is much faster with a GPU.
c:\Users\nico_\Desktop\rag_pdf\venvrag\Lib\site-packages\torch\ao\nn\quantized\dynamic\modules\rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\quantized\Quantizer.cpp:116.)
  w_ih = torch.quantize_per_tensor(


In [37]:
# Analyse des images type graphique seulement
graphique = next((image
                  for image in images_document
                  if image["type"] == "graphique"),None)

if graphique is None:
    print("Aucun graphique détecté")
else:
    print("Graphique trouvé :",graphique["chemin"])

# Lancement OCR
resultats_ocr = lecteur_ocr.readtext(graphique["chemin"],detail=1,paragraph=False)

elements_ocr = []

for  bbox_ocr, texte, confiance in resultats_ocr:

    print("Texte :", repr(texte),"| Confiance :",round(confiance, 3))

    print("Bbox dans l'image :",bbox_ocr)

    bbox_convertie = [
        [
            int(coordonnee) # pour convertir en nombre python standard
            for coordonnee in point
        ]
        for point in bbox_ocr
    ]

    element_ocr = {
        "texte": texte,
        "confiance": float(confiance), # pour convertir en nombre python standard
        "bbox_image": bbox_convertie
    }

    elements_ocr.append(
        element_ocr
    )

graphique["ocr"] = elements_ocr

print("Nombre d'éléments OCR :",len(graphique["ocr"]))

if graphique["ocr"]:
    print(graphique["ocr"][0])

texte_ocr_graphique = "\n".join(element["texte"]for element in graphique["ocr"])

graphique["texte_ocr"] = (texte_ocr_graphique)

print("\nTexte OCR complet :")
print(graphique["texte_ocr"])

Graphique trouvé : images_extraites\page_2_xref_11.png


c:\Users\nico_\Desktop\rag_pdf\venvrag\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Texte : 'Graphique 5.2.2' | Confiance : 0.451
Bbox dans l'image : [[np.int32(13), np.int32(8)], [np.int32(150), np.int32(8)], [np.int32(150), np.int32(33)], [np.int32(13), np.int32(33)]]
Texte : "Élèves possédant un télephone intelligent à l'école des Bois-Francs," | Confiance : 0.719
Bbox dans l'image : [[np.int32(14), np.int32(26)], [np.int32(568), np.int32(26)], [np.int32(568), np.int32(50)], [np.int32(14), np.int32(50)]]
Texte : 'selon le genre, 2012 à 2019' | Confiance : 0.663
Bbox dans l'image : [[np.int32(14), np.int32(44)], [np.int32(250), np.int32(44)], [np.int32(250), np.int32(68)], [np.int32(14), np.int32(68)]]
Texte : '350' | Confiance : 1.0
Bbox dans l'image : [[np.int32(37), np.int32(87)], [np.int32(63), np.int32(87)], [np.int32(63), np.int32(101)], [np.int32(37), np.int32(101)]]
Texte : '300' | Confiance : 0.999
Bbox dans l'image : [[np.int32(37), np.int32(135)], [np.int32(63), np.int32(135)], [np.int32(63), np.int32(151)], [np.int32(37), np.int32(151)]]
Texte : '250' | 

## Conversion graphique -> tableau

google/deplot

https://huggingface.co/google/deplot

In [38]:
from transformers import (AutoProcessor,Pix2StructForConditionalGeneration)

In [54]:
processeur_deplot = AutoProcessor.from_pretrained("google/deplot")

modele_deplot = (Pix2StructForConditionalGeneration.from_pretrained("google/deplot"))

modele_deplot = modele_deplot.to("cpu")

# Mode inférence
modele_deplot.eval()

c:\Users\nico_\Desktop\rag_pdf\venvrag\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nico_\.cache\huggingface\hub\models--google--deplot. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 285/285 [00:00<00:00, 381.37it/s]


Pix2StructForConditionalGeneration(
  (encoder): Pix2StructVisionModel(
    (embeddings): Pix2StructVisionEmbeddings(
      (patch_projection): Linear(in_features=768, out_features=768, bias=True)
      (row_embedder): Embedding(4096, 768)
      (column_embedder): Embedding(4096, 768)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): Pix2StructVisionEncoder(
      (layer): ModuleList(
        (0-11): 12 x Pix2StructVisionLayer(
          (attention): Pix2StructVisionAttention(
            (query): Linear(in_features=768, out_features=768, bias=False)
            (key): Linear(in_features=768, out_features=768, bias=False)
            (value): Linear(in_features=768, out_features=768, bias=False)
            (output): Linear(in_features=768, out_features=768, bias=False)
          )
          (mlp): Pix2StructVisionMlp(
            (wi_0): Linear(in_features=768, out_features=2048, bias=False)
            (wi_1): Linear(in_features=768, out_features=2048, bias=False)
 

In [55]:
image_graphique = Image.open(graphique["chemin"]).convert("RGB")

In [56]:
# Préparation de la demande
entrees = processeur_deplot(images=image_graphique,text=("Generate underlying data table of the figure below:"),return_tensors="pt")

c:\Users\nico_\Desktop\rag_pdf\venvrag\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nico_\.cache\huggingface\hub\models--ybelkada--fonts. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [57]:
# Génération du tableau
with torch.inference_mode():
    prediction = modele_deplot.generate(**entrees,max_new_tokens=512)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [58]:
# Décodage des resultats
tableau_graphique_brut = (processeur_deplot.decode(prediction[0],skip_special_tokens=True))

In [59]:
graphique["tableau_extrait"] = (tableau_graphique_brut)

In [60]:
print(graphique["tableau_extrait"])

TITLE | Graphique 5.2.2<0x0A>Élèves possèdant un télephone intelligent à l'école des Bois-Francs,<0x0A>selon le genre, 2012 à 2019<0x0A>Nombre d'élèves | Garçons | Filles<0x0A>2012 | 110 | 85<0x0A>2013 | 185 | 175<0x0A>2014 | 241 | 224<0x0A>2015 | 286 | 296<0x0A>2016 | 305 | 280<0x0A>2017 | 309 | 315<0x0A>2018 | 314 | 305<0x0A>2019 | 314 | 320


<0x0A -> indqiue qu'il passe à une autre ligne

In [79]:
graphique["tableau_extrait_brut"] = tableau_graphique_brut

tableau_graphique_nettoye = (tableau_graphique_brut.replace("<0x0A>", "\n").strip())

graphique["tableau_extrait"] = (tableau_graphique_nettoye)

print(graphique["tableau_extrait"])

TITLE | Graphique 5.2.2
Élèves possèdant un télephone intelligent à l'école des Bois-Francs,
selon le genre, 2012 à 2019
Nombre d'élèves | Garçons | Filles
2012 | 110 | 85
2013 | 185 | 175
2014 | 241 | 224
2015 | 286 | 296
2016 | 305 | 280
2017 | 309 | 315
2018 | 314 | 305
2019 | 314 | 320


In [80]:
# Transformation du tableau en dictionnaire python

lignes_deplot = [
    ligne.strip()
    for ligne
    in graphique["tableau_extrait"].splitlines() # .splitlines() sépare les textes à chaque retour de ligne
    if ligne.strip()
]

In [81]:
for ligne in lignes_deplot:
    print(ligne)

TITLE | Graphique 5.2.2
Élèves possèdant un télephone intelligent à l'école des Bois-Francs,
selon le genre, 2012 à 2019
Nombre d'élèves | Garçons | Filles
2012 | 110 | 85
2013 | 185 | 175
2014 | 241 | 224
2015 | 286 | 296
2016 | 305 | 280
2017 | 309 | 315
2018 | 314 | 305
2019 | 314 | 320


In [82]:
for index, ligne in enumerate(lignes_deplot):
    print(
        index,
        ":",
        repr(ligne),
        "| Nombre de séparateurs :",
        ligne.count("|")
    )

0 : 'TITLE | Graphique 5.2.2' | Nombre de séparateurs : 1
1 : "Élèves possèdant un télephone intelligent à l'école des Bois-Francs," | Nombre de séparateurs : 0
2 : 'selon le genre, 2012 à 2019' | Nombre de séparateurs : 0
3 : "Nombre d'élèves | Garçons | Filles" | Nombre de séparateurs : 2
4 : '2012 | 110 | 85' | Nombre de séparateurs : 2
5 : '2013 | 185 | 175' | Nombre de séparateurs : 2
6 : '2014 | 241 | 224' | Nombre de séparateurs : 2
7 : '2015 | 286 | 296' | Nombre de séparateurs : 2
8 : '2016 | 305 | 280' | Nombre de séparateurs : 2
9 : '2017 | 309 | 315' | Nombre de séparateurs : 2
10 : '2018 | 314 | 305' | Nombre de séparateurs : 2
11 : '2019 | 314 | 320' | Nombre de séparateurs : 2


On peut détecter automatiquement l'en-t^te comme la première ligne contenant au moins 2 séparateurs

### En_tête

In [83]:
index_entete = None

for index, ligne in enumerate(lignes_deplot):
    if ligne.count("|") >= 2:
        index_entete = index
        break # Arrête la boucle dès que la première ligne correspondante est trouvée

In [84]:
if index_entete is None:
    print("Aucun en-tête détecté")
else:
    print(
        "En-tête détecté :",
        lignes_deplot[index_entete]
    )

En-tête détecté : Nombre d'élèves | Garçons | Filles


In [85]:
if index_entete is not None:
    ligne_entete = lignes_deplot[index_entete]

    colonnes_graphique = [valeur.strip() for valeur in ligne_entete.split("|")]

    print(colonnes_graphique)

["Nombre d'élèves", 'Garçons', 'Filles']


### Lignes de valeurs

In [88]:
lignes_graphique_structurees = []

for ligne in lignes_deplot[index_entete + 1:]:
    valeurs = [valeur.strip() for valeur in ligne.split("|")]

    if len(valeurs) != len(colonnes_graphique):
        continue

    ligne_structuree = dict(zip(colonnes_graphique,valeurs))

    lignes_graphique_structurees.append(ligne_structuree)

In [89]:
print("Nombre de lignes structurées :", len(lignes_graphique_structurees))

for ligne in lignes_graphique_structurees:
    print(ligne)

Nombre de lignes structurées : 8
{"Nombre d'élèves": '2012', 'Garçons': '110', 'Filles': '85'}
{"Nombre d'élèves": '2013', 'Garçons': '185', 'Filles': '175'}
{"Nombre d'élèves": '2014', 'Garçons': '241', 'Filles': '224'}
{"Nombre d'élèves": '2015', 'Garçons': '286', 'Filles': '296'}
{"Nombre d'élèves": '2016', 'Garçons': '305', 'Filles': '280'}
{"Nombre d'élèves": '2017', 'Garçons': '309', 'Filles': '315'}
{"Nombre d'élèves": '2018', 'Garçons': '314', 'Filles': '305'}
{"Nombre d'élèves": '2019', 'Garçons': '314', 'Filles': '320'}


In [90]:
lignes_titre = lignes_deplot[:index_entete]

titre_graphique = " ".join(lignes_titre)

if titre_graphique.startswith("TITLE |"): # DePlot ajoute généraklement le préfixe TITLE |
    titre_graphique = (
        titre_graphique
        .split("|", 1)[1]
        .strip()
    )

In [91]:
print("Titre du graphique :",titre_graphique)

Titre du graphique : Graphique 5.2.2 Élèves possèdant un télephone intelligent à l'école des Bois-Francs, selon le genre, 2012 à 2019


In [93]:
# Regroupement des informations dans le dictionnaire graphique

graphique["donnees_structurees"] = {
    "titre": titre_graphique,
    "colonnes": colonnes_graphique,
    "lignes": lignes_graphique_structurees
}

In [94]:
print(graphique["donnees_structurees"])

{'titre': "Graphique 5.2.2 Élèves possèdant un télephone intelligent à l'école des Bois-Francs, selon le genre, 2012 à 2019", 'colonnes': ["Nombre d'élèves", 'Garçons', 'Filles'], 'lignes': [{"Nombre d'élèves": '2012', 'Garçons': '110', 'Filles': '85'}, {"Nombre d'élèves": '2013', 'Garçons': '185', 'Filles': '175'}, {"Nombre d'élèves": '2014', 'Garçons': '241', 'Filles': '224'}, {"Nombre d'élèves": '2015', 'Garçons': '286', 'Filles': '296'}, {"Nombre d'élèves": '2016', 'Garçons': '305', 'Filles': '280'}, {"Nombre d'élèves": '2017', 'Garçons': '309', 'Filles': '315'}, {"Nombre d'élèves": '2018', 'Garçons': '314', 'Filles': '305'}, {"Nombre d'élèves": '2019', 'Garçons': '314', 'Filles': '320'}]}


# Chunk

In [ ]:
premiere_image = images_document[1]

predictions = classifieur_image(
    premiere_image["chemin"],
    candidate_labels=categories,
    hypothesis_template="Cette image est {}."
)

for prediction in predictions:
    print(prediction["label"],":",round(prediction["score"],3))

a chart or graph : 0.919
a diagram or schematic : 0.061
a logo : 0.008
a scanned document : 0.006
a photograph : 0.005


## Texte

In [40]:
for numero_bloc, bloc in enumerate(textes_document):
    print(bloc)

{'type': 'texte', 'page': 1, 'content': 'TEST', 'bbox': [272.09112548828125, 50.260597229003906, 328.74444580078125, 72.6043472290039]}
{'type': 'texte', 'page': 1, 'content': 'Qu’est-ce que la génération à enrichissement contextuel ?', 'bbox': [72.0, 87.3157730102539, 280.3662414550781, 96.2532730102539]}
{'type': 'texte', 'page': 1, 'content': 'La génération à enrichissement contextuel (RAG) est le processus consistant à optimiser le résultat d’un grand modèle de', 'bbox': [72.0, 110.27476501464844, 505.14263916015625, 119.21226501464844]}
{'type': 'texte', 'page': 1, 'content': 'langage. Elle fait donc appel à une base de connaissances fiable externe aux sources de données utilisées pour l’entraîner', 'bbox': [72.0, 124.99349975585938, 509.201171875, 133.93099975585938]}
{'type': 'texte', 'page': 1, 'content': 'avant de générer une réponse. Les grands modèles de langage (LLM) sont entraînés avec d’importants volumes de données et', 'bbox': [72.0, 139.71224975585938, 525.177734375, 1

In [42]:
document_id = Path(doc.name).stem

print(document_id)

test


In [44]:
chunks_document = []

for numero_bloc, bloc in enumerate(textes_document):
    chunk_texte = {
        "chunk_id": (
            f"{document_id}"
            f"_texte_page_{bloc['page']}"
            f"_bloc_{numero_bloc}"
        ),
        "document_id": document_id,
        "type": "texte",
        "page": bloc["page"],
        "content": bloc["content"],
        "source_regions": [
            {
                "role": "texte",
                "bbox": bloc["bbox"]
            }
        ],
        "structured_data": None,
        "asset": None
    }

    chunks_document.append(
        chunk_texte
    )

In [47]:
print(chunks_document)

[{'chunk_id': 'test_texte_page_1_bloc_0', 'document_id': 'test', 'type': 'texte', 'page': 1, 'content': 'TEST', 'source_regions': [{'role': 'texte', 'bbox': [272.09112548828125, 50.260597229003906, 328.74444580078125, 72.6043472290039]}], 'structured_data': None, 'asset': None}, {'chunk_id': 'test_texte_page_1_bloc_1', 'document_id': 'test', 'type': 'texte', 'page': 1, 'content': 'Qu’est-ce que la génération à enrichissement contextuel ?', 'source_regions': [{'role': 'texte', 'bbox': [72.0, 87.3157730102539, 280.3662414550781, 96.2532730102539]}], 'structured_data': None, 'asset': None}, {'chunk_id': 'test_texte_page_1_bloc_2', 'document_id': 'test', 'type': 'texte', 'page': 1, 'content': 'La génération à enrichissement contextuel (RAG) est le processus consistant à optimiser le résultat d’un grand modèle de', 'source_regions': [{'role': 'texte', 'bbox': [72.0, 110.27476501464844, 505.14263916015625, 119.21226501464844]}], 'structured_data': None, 'asset': None}, {'chunk_id': 'test_tex

## Tableau

In [48]:
for numero_tableau, tableau in enumerate(
    tableaux_document
):
    for numero_ligne, ligne in enumerate(
        tableau["lignes"]
    ):
        contenu_ligne = " | ".join(
            f"{colonne} : {valeur}"
            for colonne, valeur
            in ligne["data"].items()
        )

        chunk_tableau = {
            "chunk_id": (
                f"{document_id}"
                f"_tableau_{numero_tableau}"
                f"_page_{tableau['page']}"
                f"_ligne_{numero_ligne}"
            ),
            "document_id": document_id,
            "type": "tableau",
            "page": tableau["page"],
            "content": contenu_ligne,
            "source_regions": [
                {
                    "role": "entete",
                    "bbox": tableau["bbox_entete"]
                },
                {
                    "role": "ligne",
                    "bbox": ligne["bbox"]
                }
            ],
            "structured_data": {
                "colonnes": tableau["colonnes"],
                "data": ligne["data"],
                "cellules": ligne["cellules"]
            },
            "asset": None
        }

        chunks_document.append(
            chunk_tableau
        )

In [49]:
print("Nombre total de chunks :", len(chunks_document))

Nombre total de chunks : 29


In [51]:
from collections import Counter

nombre_par_type = Counter(chunk["type"]for chunk in chunks_document)

print(nombre_par_type)

Counter({'tableau': 19, 'texte': 10})


In [53]:
premier_chunk_texte = next(chunk for chunk in chunks_document if chunk["type"] == "texte")

premier_chunk_tableau = next(chunk for chunk in chunks_document if chunk["type"] == "tableau")

print("Chunk texte :")
print(premier_chunk_texte)

print("\nChunk tableau :")
print(premier_chunk_tableau)

Chunk texte :
{'chunk_id': 'test_texte_page_1_bloc_0', 'document_id': 'test', 'type': 'texte', 'page': 1, 'content': 'TEST', 'source_regions': [{'role': 'texte', 'bbox': [272.09112548828125, 50.260597229003906, 328.74444580078125, 72.6043472290039]}], 'structured_data': None, 'asset': None}

Chunk tableau :
{'chunk_id': 'test_tableau_0_page_1_ligne_0', 'document_id': 'test', 'type': 'tableau', 'page': 1, 'content': 'a : 1 | b : 1 | c : 1', 'source_regions': [{'role': 'entete', 'bbox': [72.5, 238.5, 297.5, 255.5]}, {'role': 'ligne', 'bbox': [72.5, 255.5, 297.5, 271.5]}], 'structured_data': {'colonnes': ['a', 'b', 'c'], 'data': {'a': '1', 'b': '1', 'c': '1'}, 'cellules': [{'colonne': 'a', 'valeur': '1', 'bbox': [72.5, 255.5, 147.5, 271.5]}, {'colonne': 'b', 'valeur': '1', 'bbox': [147.5, 255.5, 222.5, 271.5]}, {'colonne': 'c', 'valeur': '1', 'bbox': [222.5, 255.5, 297.5, 271.5]}]}, 'asset': None}
